In [ ]:
from sklearn.datasets import load_sample_images
import tensorflow as tf
import matplotlib.pyplot as plt

In [ ]:
images = load_sample_images()['images']
plt.figure(figsize=(15, 10))
plt.subplot(121)
plt.imshow(images[0])
plt.axis('off')
plt.subplot(122)
plt.imshow(images[1])
plt.axis('off')

In [ ]:
images = tf.keras.layers.CenterCrop(height=70, width=120)(images)
images = tf.keras.layers.Rescaling(scale=1/255.)(images)
plt.figure(figsize=(12, 8))
plt.subplot(121)
plt.imshow(images[0])
plt.axis('off')
plt.subplot(122)
plt.imshow(images[1])
plt.axis('off');

In [ ]:
images.shape

In [ ]:
tf.random.set_seed(42)
conv_layer = tf.keras.layers.Conv2D(filters=32, kernel_size=7)
fmaps = conv_layer(images)

In [ ]:
fmaps.shape

In [ ]:
plt.figure(figsize=(15, 9))
for image_idx in (0, 1):
  for fmap_idx in (0, 1):
    plt.subplot(2, 2, image_idx * 2 + fmap_idx + 1)
    plt.imshow(fmaps[image_idx, :, :, fmap_idx], cmap='gray')
    plt.axis('off')

plt.show()

In [ ]:
conv_layer = tf.keras.layers.Conv2D(filters=32, kernel_size=7, padding='same')

fmaps = conv_layer(images)

In [ ]:
fmaps.shape

In [ ]:
conv_layer = tf.keras.layers.Conv2D(filters=32, kernel_size=7, padding='same', strides=2)

fmaps = conv_layer(images)
fmaps.shape

In [ ]:
kernels, biases = conv_layer.get_weights()
kernels.shape

In [ ]:
tf.random.set_seed(42)
filters = tf.random.normal([7, 7, 3, 2])
biases = tf.zeros([2])
fmaps = tf.nn.conv2d(images, filters, strides=1, padding='SAMe') + biases

17/03/2025

# Pooling Layers

## Implementing Pooling Layers with Keras
## MaxPooling

In [ ]:
max_pool = tf.keras.layers.MaxPool2D(pool_size=2)

In [ ]:
output = max_pool(images)

In [ ]:
import matplotlib as mpl

fig = plt.figure(figsize=(12, 8))
gs = mpl.gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[2, 1])

ax1 = fig.add_subplot(gs[0, 0])
ax1.set_title('Input')
ax1.imshow(images[0])
ax1.axis('off')
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_title('Output')
ax2.imshow(output[0])
ax2.axis('off')
plt.show()

In [ ]:
global_avg_pool = tf.keras.layers.GlobalAvgPool2D()

In [ ]:
images.shape

In [ ]:
global_avg_pool = tf.keras.layers.Lambda(
    lambda X: tf.reduce_mean(X, axis=[1, 2])
)

In [ ]:
global_avg_pool(images)

In [ ]:
import tensorflow as tf

In [ ]:
import numpy as np

In [ ]:
mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = mnist
X_train_full = np.expand_dims(X_train_full, axis=-1).astype(np.float32) / 255
X_test = np.expand_dims(X_test.astype(np.float32), axis=-1) / 255
X_train, X_valid = X_train_full[:-5000], X_train_full[-5000:]
y_train, y_valid = y_train_full[:-5000], y_train_full[-5000:]


In [ ]:
X_train_full.shape

In [ ]:
X_train.shape

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape = [28, 28, 1]),

    tf.keras.layers.Conv2D(filters=64, kernel_size=7, padding='same', activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.MaxPool2D(),

    tf.keras.layers.Conv2D(filters=128, kernel_size=3, padding='same', activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Conv2D(filters=128, kernel_size=3, padding='same', activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.MaxPool2D(),

    tf.keras.layers.Conv2D(filters=256, kernel_size=3, padding='same', activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Conv2D(filters=256, kernel_size=3, padding='same', activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.MaxPool2D(),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(units=128, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(units=64, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(units=10, activation='softmax')
])

In [ ]:
model.compile(loss='sparse_categorical_crossentropy',
              optimizer='nadam',
              metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=10,
                    validation_data=(X_valid, y_valid))

score=model.evaluate(X_test, y_test)
X_new = X_test[:10]
y_pred = model.predict(X_new)

In [ ]:
model.summary()

## ResNet34

In [ ]:
from functools import partial

In [ ]:
DefaultConv2D = partial(tf.keras.layers.Conv2D, kernel_size=3, strides=1,
                        padding='same', kernel_initializer='he_normal',
                        use_bias=False)

class ResidualUnit(tf.keras.layers.Layer):
  def __init__(self, filters, strides=1, activation='relu', **kwargs):
    super().__init__(**kwargs)
    self.activation = tf.keras.activations.get(activation)
    self.main_layers = [
        DefaultConv2D(filters, strides=strides),
        tf.keras.layers.BatchNormalization(),
        self.activation,
        DefaultConv2D(filters),
        tf.keras.layers.BatchNormalization()
    ]

    self.skip_layers = []
    if strides > 1:
      self.skip_layers = [
          DefaultConv2D(filters, kernel_size=1, strides=strides),
          tf.keras.layers.BatchNormalization()
      ]

  def call(self, inputs):
    Z = inputs
    for layer in self.main_layers:
      Z = layer(Z)
    skip_Z = inputs
    for layer in self.skip_layers:
      skip_Z = layer(skip_Z)
    return self.activation(Z + skip_Z)

# Using Pretrained Models from Keras

In [ ]:
model = tf.keras.applications.ResNet50(weights='imagenet')

In [ ]:
images = load_sample_images()["images"]

images_stacked = tf.stack(images, axis=0)

images_resized = tf.keras.layers.Resizing(height=224, width=224,
                                          crop_to_aspect_ratio=True)(images_stacked)

In [ ]:
inputs = tf.keras.applications.resnet50.preprocess_input(images_resized)

In [ ]:
inputs = tf.cast(inputs, tf.float32)

In [ ]:
Y_proba = model.predict(inputs)
Y_proba.shape

In [ ]:
top_K = tf.keras.applications.resnet50.decode_predictions(Y_proba, top=3)
for image_index in range(len(images)):
  print(f"Image #{image_index}")
  for class_id, name, y_proba in top_K[image_index]:
    print(f"{class_id} - {name:12s} {y_proba:.2%}")

# Pretrained Models for Transfer Learning

In [ ]:
import tensorflow_datasets as tfds

dataset, info = tfds.load("tf_flowers", as_supervised=True, with_info=True)
dataset_size = info.splits["train"].num_examples
class_names = info.features["label"].names
n_classes = info.features["label"].num_classes

In [ ]:
test_set_raw, valid_set_raw, train_set_raw = tfds.load(
    "tf_flowers",
    split=["train[:10%]", "train[10%:25%]", "train[25%:]"],
    as_supervised=True
)

In [ ]:
train_set_raw

28/03/2025

In [ ]:
import matplotlib.pyplot as plt

index = 0
plt.figure(figsize=(12, 10))
for image, label in train_set_raw.take(9):
  index += 1
  plt.subplot(3, 3, index)
  plt.imshow(image)
  plt.axis('off')
  plt.title(class_names[label])

plt.show()

In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

batch_size = 32

preprocessing = tf.keras.Sequential([
    tf.keras.layers.Resizing(height=224, width=224,
                             crop_to_aspect_ratio=True),
    tf.keras.layers.Lambda(tf.keras.applications.xception.preprocess_input)
])

# tf.data.AUTOTUNE

train_set = train_set_raw.map(lambda X, y: (preprocessing(X), y))
train_set = train_set.shuffle(1000).batch(batch_size).prefetch(1)
valid_set = valid_set_raw.map(lambda X, y: (preprocessing(X), y)).batch(batch_size)
test_set = test_set_raw.map(lambda X, y: (preprocessing(X), y)).batch(batch_size)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip(mode="horizontal", seed=42),
    tf.keras.layers.RandomRotation(factor=0.05, seed=42),
    tf.keras.layers.RandomContrast(factor=0.2, seed=42)
])

In [ ]:
import numpy as np

plt.figure(figsize=(12, 10))
for X_batch, y_batch in train_set.take(1):
  for index in range(9):
    plt.subplot(3, 3, index + 1)
    plt.imshow(X_batch[index] / 2 + 0.5)
    plt.axis('off')
    plt.title(class_names[y_batch[index]])

In [ ]:
data_augmentation_idk = tf.keras.Sequential([
    tf.keras.layers.RandomFlip(mode="horizontal", seed=42),
    tf.keras.layers.RandomRotation(factor=0.05, seed=42),
    # tf.keras.layers.RandomContrast(factor=0.2, seed=42)
])

import numpy as np

plt.figure(figsize=(12, 10))
for X_batch, y_batch in train_set.take(1):
  X_batch_augmented = data_augmentation_idk(X_batch)
  for index in range(9):
    plt.subplot(3, 3, index + 1)
    plt.imshow(X_batch_augmented[index] / 2 + 0.5)
    plt.axis('off')
    plt.title(class_names[y_batch[index]])

In [ ]:
tf.random.set_seed(42)
base_model = tf.keras.applications.xception.Xception(weights='imagenet',
                                                     include_top=False)

avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output = tf.keras.layers.Dense(n_classes, activation='softmax')(avg)
model = tf.keras.Model(inputs=base_model.input, outputs=output)

In [ ]:
for layer in base_model.layers:
  layer.trainable = False

In [ ]:
model.compile(loss = 'sparse_categorical_crossentropy',
              optimizer = tf.keras.optimizers.SGD(learning_rate=0.1, momentum=0.9),
              metrics = ['accuracy'])
# history =  model.fit()....

In [ ]:
for indices in zip(range(33), range(33, 66), range(66, 99), range(99, 132)):
  for idx in indices:
    print(f"{idx:3}: {base_model.layers[idx].name:22}", end="")
  print()

In [ ]:
for layer in base_model.layers:
  print(layer.name)

In [ ]:
for layer in base_model.layers[56:]:
  layer.trainable = True

model.compile(loss='sparse_categorical_crossentropy',
              optimizer=tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
              metrics=['accuracy'])

# history = model.fit(train_set, validation_data=valid_set, epochs=10).....